# Silver — Transactions (SCD1)
**GlobalMart Orchestration Lab**

| | |
|---|---|
| **Source** | `{catalog}.bronze.transactions` |
| **Target** | `{catalog}.silver.transactions` |
| **SCD Type** | SCD1 — financial records update in place |
| **Depends on** | Bronze Transactions **and** Silver Orders (referential-integrity check) |

## Step 1 — Setup

In [ ]:
from pyspark.sql.functions import col, row_number, desc, current_timestamp, when, lit
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dbutils.widgets.text('catalog',       'your_catalog')
dbutils.widgets.text('source_schema', 'bronze')
dbutils.widgets.text('target_schema', 'silver')

CATALOG       = dbutils.widgets.get('catalog')
SOURCE_SCHEMA = dbutils.widgets.get('source_schema')
TARGET_SCHEMA = dbutils.widgets.get('target_schema')

SOURCE_TABLE  = f'{CATALOG}.{SOURCE_SCHEMA}.transactions'
SILVER_ORDERS = f'{CATALOG}.{TARGET_SCHEMA}.orders'
TABLE         = f'{CATALOG}.{TARGET_SCHEMA}.transactions'

print(f'Source: {SOURCE_TABLE}')
print(f'Target: {TABLE}')

## Step 2 — Read &amp; Deduplicate from Bronze

In [ ]:
bronze_df = spark.table(SOURCE_TABLE)
print(f"Bronze rows (all batches): {bronze_df.count():,}")

dedup_window = Window.partitionBy("transaction_id").orderBy(desc("_ingested_at"))
deduped_df = (
    bronze_df
    .withColumn("_rn", row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)
print(f"After dedup (latest per transaction_id): {deduped_df.count():,}")

## Step 3 — DQ Scan
Bronze's Autoloader read uses `cloudFiles.inferColumnTypes=true`, but casting explicitly here still matters — it's the contract Silver guarantees downstream, independent of whatever Autoloader happened to infer from a given batch's file. Negative `total_amount` is checked for; a genuinely negative amount could be a valid refund/return in a real system, but for this lab dataset it signals a broken row.

In [ ]:
casted_df = (
    deduped_df
    .withColumn("quantity",     col("quantity").cast("int"))
    .withColumn("unit_price",   col("unit_price").cast("double"))
    .withColumn("total_amount", col("total_amount").cast("double"))
)

known_orders = spark.table(SILVER_ORDERS).select("order_id").distinct()

dq_df = casted_df.join(
    known_orders.withColumnRenamed("order_id", "_known_order_id"),
    casted_df.order_id == col("_known_order_id"),
    "left"
).withColumn(
    "_dq_issue",
    when(col("transaction_id").isNull(),          lit("NULL_TRANSACTION_ID"))
    .when(col("_known_order_id").isNull(),         lit("ORPHANED_ORDER_ID"))
    .when(col("total_amount") < 0,                 lit("NEGATIVE_TOTAL_AMOUNT"))
    .otherwise(lit(None))
).drop("_known_order_id")

dq_df.groupBy("_dq_issue").count().orderBy(desc("count")).display()

## Step 4 — Decision Per Issue

| Issue | Decision | Why |
|---|---|---|
| `NULL_TRANSACTION_ID` | **Quarantine** | No usable key |
| `ORPHANED_ORDER_ID` | **Quarantine** | Can't attribute the transaction to a real order |
| `NEGATIVE_TOTAL_AMOUNT` | **Quarantine** | Not a valid financial record for this lab's data — investigate before assuming this in a real system, since a real negative amount can be a legitimate refund |

In [ ]:
quarantine_df = dq_df.filter(col("_dq_issue").isNotNull())
clean_df      = dq_df.filter(col("_dq_issue").isNull()).drop("_dq_issue")

print(f"Quarantined : {quarantine_df.count():,}")
print(f"Valid transactions: {clean_df.count():,}")
if quarantine_df.count() > 0:
    quarantine_df.select("transaction_id", "order_id", "total_amount").display()

## Step 5 — Create Silver Table (first run only)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TARGET_SCHEMA}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        transaction_id     STRING,
        order_id           STRING,
        quantity           INT,
        unit_price         DOUBLE,
        total_amount       DOUBLE,
        payment_mode       STRING,
        created_at         STRING,
        _silver_updated_at TIMESTAMP
    )
    USING DELTA
""")
print(f"Table ready: {TABLE}")

## Step 6 — SCD1 MERGE
Update if `transaction_id` already exists, insert if new. Safe to re-run.

In [ ]:
silver_df = clean_df.withColumn("_silver_updated_at", current_timestamp()) \
    .select("transaction_id", "order_id", "quantity", "unit_price", "total_amount", "payment_mode", "created_at", "_silver_updated_at")

target = DeltaTable.forName(spark, TABLE)

(target.alias("t")
    .merge(silver_df.alias("s"), "t.transaction_id = s.transaction_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("MERGE complete")

## Step 7 — Verify

In [ ]:
result = spark.table(TABLE)
print(f"Total rows in {TABLE}: {result.count():,}")
print("Payment mode distribution:")
result.groupBy("payment_mode").count().display()
result.display()